# Signal research

Our workspace for new signals, from "I have an idea" to "it's a sleeve in the portfolio". Every step calls the same
code as the research scripts, the portfolio backtest and the runtime, so a signal that passes here is the signal that
would trade.

**The workflow.** Each section is one step. Run it top to bottom once, then change sections 2 and 3 to your idea and
run it again.

1. **Data**: Kraken perpetuals from 2020 (BTC and ETH), cached locally.
2. **The idea**: does the feature predict returns at all, before any trading rules? (IC, buckets, by year)
3. **The strategy**: turn the feature into positions with `rule_strategy` or `signal_strategy`.
4. **One backtest** with perp costs, in-sample vs holdout, against buy-and-hold.
5. **Scorecard**: holdout, buy-and-hold, costs, a placebo test of the timing, consistency by year, then pass/fail.
6. **Robustness**: a parameter sweep. A plateau is real; a single bright cell is luck.
7. **Portfolio fit**: does it improve the book in `config/portfolio.example.toml`, and with which sizing?
8. **Record and promote.**

Appendix A covers many coins at once (the Binance archive). Appendix B loads results saved by the CLI.

Read `docs/research_guide.md` once, for the method and how to read the numbers. Past findings, failures included, are
in `docs/research_log.md`. The older `strategy_research.ipynb` works on Kraken spot, which has only 720 candles.

Kernel: `Python 3 (CryptoArb)`.

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # data/ and config/ paths resolve the same way as for the CLI

import numpy as np
import pandas as pd

from src.backtest.indicators import atr, rolling_max, rolling_mean, rolling_min, rolling_std, rsi, shift
from src.backtest.strategies import rule_strategy, signal_strategy
from src.portfolio.backtest import candidate_report
from src.portfolio.config import load_portfolio_config
from src.portfolio.sleeves import SleeveSpec
from src.research import CostSettings, catalog_table, load_bars, plot_heatmap, plot_run, run_strategy, summarize, sweep
from src.research.features import bars_frame, bucket_table, forward_return, ic_by_year, information_coefficient, past_return, zscore
from src.research.scorecard import CHECKS, checks, scorecard

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

COSTS = CostSettings.perp()  # Kraken Futures: 0.05% taker + 5 bps slippage per fill, funding 0.01%/day paid by longs

## 1. Data

Kraken Futures perpetuals, BTC/USD and ETH/USD from 2020-02-26 (the inverse contract until 2022, the linear one
after). They are cached in `data/historical_cache/`, and only candles newer than the cache are downloaded. Intervals:
`"1d"`, `"4h"`, `"1h"`, `"15m"`. For spot, use `load_bars("BTC/EUR", "1d")` (the last 720 candles only).

In [ ]:
SYMBOLS = ["BTC/USD", "ETH/USD"]
INTERVAL = "1d"

data = {symbol: load_bars(symbol, INTERVAL, source="perp") for symbol in SYMBOLS}
frames = {symbol: bars_frame(bars) for symbol, bars in data.items()}
for symbol, bars in data.items():
    print(f"{symbol}: {len(bars)} {INTERVAL} bars, {bars[0].timestamp:%Y-%m-%d} -> {bars[-1].timestamp:%Y-%m-%d}")

**What exists already.** Before writing something new, check that it isn't one of these. Also check
`docs/research_log.md` for what has been tried.

In [ ]:
catalog_table()[["strategy", "family", "hypothesis", "defaults"]]

## 2. The idea: does the feature predict anything?

A strategy is a feature, plus trading rules, plus costs. Test the feature on its own first. Measure it at each bar's
close (using data up to that close only), then look at what the price did *after* it:

- `information_coefficient`: the rank correlation between the feature and the forward return. On daily bars, |IC|
  of 0.02-0.05 is already useful, if the sign is stable.
- `bucket_table`: the mean forward return in bps per feature quintile, with a t-stat on non-overlapping bars. Compare
  the spread between the top and bottom buckets with the round-trip cost (about 20 bps taker on perps).
- `ic_by_year`: a real effect keeps its sign year after year.

The example feature is 20-day momentum, compared with its own last year (a z-score). Replace `feature` with your idea.

In [ ]:
def feature(frame: pd.DataFrame) -> np.ndarray:
    """The idea as one number at each bar's close, from data up to that close only."""
    return zscore(past_return(frame["close"].to_numpy(), 20), 365)


HORIZONS = [1, 5, 10, 20]  # bars ahead
ic = pd.DataFrame(
    {symbol: [information_coefficient(feature(frame), forward_return(frame["close"], horizon)) for horizon in HORIZONS] for symbol, frame in frames.items()},
    index=pd.Index(HORIZONS, name="horizon"),
)
ic.round(3)

In [ ]:
frame = frames["BTC/USD"]
bucket_table(feature(frame), forward_return(frame["close"], 10), horizon=10)

In [ ]:
pd.DataFrame({symbol: ic_by_year(feature(frame), forward_return(frame["close"], 10), frame.index) for symbol, frame in frames.items()}).round(3)

## 3. The strategy

Turn the feature into the position to hold after each close: 1 is long, -1 is short, 0 is flat. There are two ways to
do it. Both give a strategy that works in backtests, sweeps, the scorecard and as a portfolio sleeve, and both are
vectorized (years of bars in milliseconds).

- `rule_strategy(rules, warmup=...)`: entry and exit rules. `rules(bars)` returns
  `(long_entry, long_exit, short_entry, short_exit)` boolean arrays, with `None, None` for a long-only idea. A trade
  stays open until its exit rule fires.
- `signal_strategy(signals, warmup=...)`: the position directly, e.g. `np.sign(z)`.

`bars` has `.close`, `.high`, `.low` and `.volume` arrays. Use the causal indicators in `src.backtest.indicators`
(`rolling_mean`, `rolling_max`, `shift`, `atr`, `rsi`, ...): element t only uses bars up to t. `warmup` is the number
of bars needed before the first signal.

Wrap the strategy in a factory, a function of its parameters, so section 6 can sweep it.

In [ ]:
def momentum_z(lookback=20, z_window=365, entry=0.5, exit=0.0):
    """Long while momentum is unusually strong for this coin, short while unusually weak; out when it fades."""

    def rules(bars):
        z = zscore(past_return(bars.close, lookback), z_window)
        return z > entry, z < exit, z < -entry, z > -exit

    return rule_strategy(rules, warmup=z_window + lookback + 1)


NAME = "momentum_z"
PARAMS = {"lookback": 20, "z_window": 365, "entry": 0.5, "exit": 0.0}
WARMUP = PARAMS["z_window"] + PARAMS["lookback"] + 1
strategy = momentum_z(**PARAMS)

# The same idea as a direct position, for comparison:
# strategy = signal_strategy(lambda bars: np.sign(zscore(past_return(bars.close, 20), 365)), warmup=WARMUP)

## 4. One backtest

`run_strategy` runs one continuous backtest. Every fill pays costs, the position is full size (100% of equity) and
there is no risk overlay, so the numbers measure the signal itself. In-sample is the first 70% of bars, and holdout
is the last 30%. `buy_hold_*` is simply holding the coin over the same bars. `measure_start=WARMUP` starts the
in-sample period when the strategy can first trade.

In [ ]:
run = run_strategy(data["BTC/USD"], strategy, costs=COSTS, measure_start=WARMUP, label=NAME)
run.metrics_table()

In [ ]:
plot_run(run);

## 5. Scorecard

The scorecard gives one row per symbol with every check a new signal must pass
(`src/research/scorecard.py`):

- `is_sharpe` and `ho_sharpe`: in-sample and holdout. The holdout checks the in-sample result; it isn't where you tune.
- `is_buy_hold_sharpe`: holding the coin over the same bars. A long-biased trend rule in a bull market has to beat it.
- `is_sharpe_maker`, `is_sharpe_zero_cost`, `is_sharpe_funding_x3`: the same run with limit-order fees, with no costs,
  and with triple funding. If only the zero-cost run is good, the signal is real but not tradable at our fees.
- `placebo_beaten_is`: the share of time-shifted copies of the same positions that it beats (same trades, holding
  times and exposure, wrong dates). About 0.5 means the timing adds nothing and the result comes from exposure alone.
  0.9 or more means the timing is real.
- `years_positive`, `worst_year`: consistency by calendar year.

Always score a benchmark from the same start date next to the new idea. Here it is the MA(4,48) long-only rule, which
is already a sleeve in the book.

In [ ]:
card = pd.concat([
    scorecard(data, strategy, measure_start=WARMUP, label=NAME, costs=COSTS),
    scorecard(data, "moving_average_crossover", params={"short_window": 4, "long_window": 48, "long_only": True}, measure_start=WARMUP, costs=COSTS),
], ignore_index=True)
card.round(2)

In [ ]:
for name, meaning in CHECKS.items():
    print(f"{name}: {meaning}")
checks(card)

**Other things to try here.** `long_only` versions (wrap the strategy with `make_long_only` from
`src.backtest.strategies`). 4h bars, with windows scaled by 6. `CostSettings.perp(maker=True)` for resting limit
orders.

## 6. Robustness: sweep the parameters

`sweep` runs every combination in the grid on every symbol, both long-only and long/short. `summarize` gives one
verdict per side, and `plot_heatmap` shows in-sample next to holdout. A robust result looks like a **broad block of
one colour in-sample that keeps its colour in the holdout**, with `best_neighbors_is` close to `best_is`. A single
bright cell is luck. Pick parameters from the middle of a plateau, never the best cell.

Every combination is measured from the same bar (the largest warmup in the grid), so their results line up.

In [ ]:
GRID = {"lookback": [10, 20, 40, 60], "entry": [0.25, 0.5, 1.0], "exit": [0.0, 0.25]}
SWEEP_START = PARAMS["z_window"] + max(GRID["lookback"]) + 1

results = sweep(data, momentum_z, grid=GRID, costs=COSTS, measure_start=SWEEP_START)
summarize(results)

In [ ]:
plot_heatmap(results, NAME, x="lookback", y="entry", long_only=False);

## 7. Portfolio fit

A signal is worth trading when it makes the book better, not when it looks good alone. A sleeve that repeats what
the book already holds adds risk without adding return. `candidate_report` adds the idea as a sleeve to the example
portfolio (`config/portfolio.example.toml`: five trend sleeves on the BTC and ETH perps). It runs the book with and
without it, under the config's allocation and risk limits, and shows the new sleeve's correlation with each existing
one. Look for low correlation and a better book in **both** periods: higher Sharpe or lower drawdown.

It also compares the two usual sizings. `fixed_fraction` takes the same share of equity every trade. `vol_target`
sizes each entry to a target volatility, which is how the existing sleeves are sized. The candidate's warmup can move
the start date, so the "without" book here can differ slightly from `docs/research_log.md`.

In [ ]:
book = load_portfolio_config("config/portfolio.example.toml")
SIZINGS = {"fixed_fraction": {"fraction": 1.0}, "vol_target": {"target_annual_vol": 0.5}}

reports = {}
for sizing, sizing_params in SIZINGS.items():
    candidate = SleeveSpec(id=f"btc_{NAME}", instrument="kraken_futures:BTC/USD", interval=INTERVAL, strategy=NAME,
                           sizing=sizing, sizing_params=sizing_params, warmup_bars=WARMUP)
    reports[sizing] = candidate_report(book, candidate, strategy=strategy)

columns = ["book", "period", "sharpe", "cagr", "max_drawdown", "avg_gross_exposure"]
pd.concat({sizing: pd.concat([report["alone"], report["books"]])[columns] for sizing, report in reports.items()}).round(2)

In [ ]:
reports["vol_target"]["correlation"].T.round(2)

## 8. Record and promote

1. Write the result in `docs/research_log.md`, even when it fails: the idea, data, costs, IS/HO, scorecard, sweep
   verdict and portfolio fit.
2. If it earns a place, move the factory into `src/backtest/strategies.py`, register it in `StrategyRegistry`
   (`src/backtest/runner.py`) and add a `StrategySpec` (hypothesis, defaults, grid) in `src/research/catalog.py`.
   From then on it works by name everywhere, including the runtime's `--strategy`.
3. Add a `[[sleeves]]` block to a portfolio config and backtest the whole book:
   `python scripts/research/portfolio_backtest.py config/portfolio.example.toml`.
4. Paper-run it before any real money.

## Appendix A: many coins at once (the Binance archive)

Cross-sectional ideas (rank the coins, go long the best and short the worst) need many coins. `update_cache()` in
`src/data/binance_archive.py` downloads daily bars and funding for every USDT perpetual from Binance's public archive
into `data/historical_cache/binance_um/`. It includes delisted coins, so there is no survivorship bias.
`scripts/research/cross_sectional_study.py` is the full study. The cell below is the short version: one feature, its
daily IC across coins, and a weekly long/short book of the top and bottom 20%.

In [ ]:
from src.data.binance_archive import load_panel
from src.research.portfolio import PortfolioCosts, cross_sectional_ic, liquid_universe, rank_weights, simulate_portfolio

klines = load_panel("klines_1d")
if klines.empty:
    print("No Binance archive cached yet: run `python scripts/research/cross_sectional_study.py --download` (about 50,000 small files).")
else:
    close = klines.pivot_table(index="date", columns="symbol", values="close")
    volume = klines.pivot_table(index="date", columns="symbol", values="quote_volume")
    close = close.where(volume > 0)                  # a day without trades has no usable price
    universe = liquid_universe(volume, top_n=50)     # the 50 most traded coins each day, from past volume only
    score = close / close.shift(30) - 1.0            # the feature: 30-day momentum per coin
    forward = close.shift(-7) / close - 1.0          # what happened over the next 7 days
    daily_ic = cross_sectional_ic(score, forward, universe)
    print(f"{close.shape[1]} coins; mean daily IC {daily_ic.mean():.3f}, days positive {(daily_ic > 0).mean():.0%}")
    display(daily_ic.groupby(daily_ic.index.year).mean().round(3).to_frame("mean IC by year"))
    long_short = simulate_portfolio(close, rank_weights(score, universe, quantile=0.2), costs=PortfolioCosts(fee_pct=0.05, slippage_bps=10.0), rebalance_every=7)
    display(pd.DataFrame({"in-sample": long_short.metrics(end="2024-01-01"), "holdout": long_short.metrics(start="2024-01-01")}).round(3))

## Appendix B: results saved by the CLI

`python scripts/research/research.py sweep ...` writes `results.csv` and `summary.csv` to `data/research/<run>/`.
Load them here to slice them however you like.

In [ ]:
saved_runs = sorted(Path("data/research").glob("sweep_*/results.csv"), key=lambda path: path.stat().st_mtime)
if saved_runs:
    saved = pd.read_csv(saved_runs[-1])
    print(f"Loaded {saved_runs[-1]} ({len(saved)} rows)")
    display(summarize(saved[saved["interval"] == "1d"]).head(10))
else:
    print("No saved sweeps yet: run `python scripts/research/research.py sweep`")